# 01 - EGFR

In [2]:
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=Path(__file__).resolve().parent.parent / ".env" if "__file__" in dir() else Path("../.env"), override=True)
data_dir = Path(os.getenv("FILE_PATH_GROUP_A"))
print(f"data_dir: {data_dir.resolve()}, exists: {data_dir.resolve().exists()}")
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

columns_to_keep = [
    "Sample", "Name", "TPM", "RPKM",
    "Unique gene reads", "Total gene reads",
    "Transcripts annotated", "Uniquely identified transcripts",
    "Unique exon reads", "Total exon reads",
    "Ratio of unique to total (exon reads)",
    "Unique exon-exon reads", "Total exon-exon reads",
    "Unique intron reads", "Total intron reads",
    "Ratio of intron to total gene reads",
]

ge_files = sorted(data_dir.glob("*(GE).xlsx"))
print(f"Found {len(ge_files)} GE file(s):\n")

all_egfr = []

for filepath in ge_files:
    sample_id = filepath.stem.replace(" (GE)", "")
    print(f"  Processing: {filepath.name}  →  Sample ID: {sample_id}")

    df = pd.read_excel(filepath)
    df_egfr = df[df["Name"] == "EGFR"].copy()
    df_egfr.insert(0, "Sample", sample_id)

    individual_path = output_dir / f"{sample_id}_EGFR_only.xlsx"
    df_egfr.to_excel(individual_path, index=False)

    all_egfr.append(df_egfr)

df_all = pd.concat(all_egfr, ignore_index=True)
df_all = df_all[[c for c in columns_to_keep if c in df_all.columns]]

print(f"\nAggregated EGFR rows: {len(df_all)}")
display(df_all)

agg_path = output_dir / "EGFR_all_samples_a.xlsx"
df_all.to_excel(agg_path, index=False)
print(f"\nSaved aggregated file to: {agg_path}")

data_dir: D:\Projects\Emilia\data, exists: True
Found 1 GE file(s):

  Processing: R22103005-1016330242_S10 (GE).xlsx  →  Sample ID: R22103005-1016330242_S10

Aggregated EGFR rows: 1


,Sample,Name,TPM,RPKM,Unique gene reads,Total gene reads,Transcripts annotated,Uniquely identified transcripts,Unique exon reads,Total exon reads,Ratio of unique to total (exon reads),Unique exon-exon reads,Total exon-exon reads,Unique intron reads,Total intron reads,Ratio of intron to total gene reads
0,R22103005-1016330242_S10,EGFR,176901.202317,19558.96273,732206,735431,11,9,478885,482094,0.993344,332864,332890,253321,253337,0.344474



Saved aggregated file to: ..\data\processed\EGFR_all_samples_a.xlsx


## Aggregate by Group

Combine all `EGFR_all_samples_<group>.xlsx` files into a single file with a `Gruppe` column.

In [3]:
output_dir = Path("../data/processed")
group_files = sorted(output_dir.glob("EGFR_all_samples_*.xlsx"))
print(f"Found {len(group_files)} group file(s):\n")

all_groups = []

for filepath in group_files:
    group_label = filepath.stem.replace("EGFR_all_samples_", "").upper()
    print(f"  {filepath.name}  →  Gruppe: {group_label}")

    df_group = pd.read_excel(filepath)
    df_group.insert(0, "Gruppe", group_label)
    all_groups.append(df_group)

df_combined = pd.concat(all_groups, ignore_index=True)

print(f"\nCombined rows: {len(df_combined)}")
print(f"Groups: {df_combined['Gruppe'].value_counts().to_dict()}")
display(df_combined)

combined_path = output_dir / "EGFR_combined.xlsx"
df_combined.to_excel(combined_path, index=False)
print(f"\nSaved to: {combined_path}")

Found 1 group file(s):

  EGFR_all_samples_a.xlsx  →  Gruppe: A

Combined rows: 1
Groups: {'A': 1}


,Gruppe,Sample,Name,TPM,RPKM,Unique gene reads,Total gene reads,Transcripts annotated,Uniquely identified transcripts,Unique exon reads,Total exon reads,Ratio of unique to total (exon reads),Unique exon-exon reads,Total exon-exon reads,Unique intron reads,Total intron reads,Ratio of intron to total gene reads
0,A,R22103005-1016330242_S10,EGFR,176901.202317,19558.96273,732206,735431,11,9,478885,482094,0.993344,332864,332890,253321,253337,0.344474



Saved to: ..\data\processed\EGFR_combined.xlsx
